# Clase 13: Aprendizaje de Máquina (Supervisado)


Recordemos: Aprendizaje a partir de datos previamente etiquetados $$x_i\to y_i.$$ Las etiquetas pueden ser categóricas o continuas (regresión).

Para el primer ejercicio vamos a hacer una regresión lineal para ajustar datos históricos de temperatura.


In [ ]:
import pandas as pd
import requests
from io import StringIO

ESTACION_ELDORADO = "80222099999"  # ID global de la estación (WMO)
AÑO_INICIO = 1965
AÑO_FIN = 2025

# Base URL del repositorio GSOD de la NOAA (formato CSV)
BASE_URL = "https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/"

datos_completos = []

print("Iniciando descarga de datos desde la NOAA...")

for anio in range(AÑO_INICIO, AÑO_FIN + 1):
    url = f"{BASE_URL}{anio}/{ESTACION_ELDORADO}.csv"

    try:
        # Realizamos la petición para el año específico
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            # Leemos el CSV directamente desde la respuesta en memoria
            df_anio = pd.read_csv(StringIO(response.text))
            datos_completos.append(df_anio)
            print(f"Año {anio}: Descargado con éxito.")
        elif response.status_code == 404:
            print(f"Año {anio}: No se encontraron datos (404).")
        else:
            print(f"Año {anio}: Error en el servidor (Código {response.status_code}).")

    except Exception as e:
        print(f"Año {anio}: Error en la conexión -> {e}")

# Concatenamos todos los años en un solo DataFrame
if datos_completos:
    df_total = pd.concat(datos_completos, ignore_index=True)

    # Convertir columna de fecha a formato datetime
    df_total['DATE'] = pd.to_datetime(df_total['DATE'])

    # --- PROCESAMIENTO Y CONVERSIÓN ---
    # La NOAA entrega la temperatura en Fahrenheit ('TEMP'). La convertimos a Celsius.
    df_total['TEMP_C'] = (df_total['TEMP'] - 32) * 5 / 9

    # Agrupamos por año y mes para obtener la media mensual que solicitaste
    df_total['AÑO_MES'] = df_total['DATE'].dt.to_period('M')

    # Extraemos el promedio mensual de la temperatura media, máxima y mínima
    # Nota: MAX y MIN también vienen originalmente en Fahrenheit.
    df_total['MAX_C'] = (df_total['MAX'] - 32) * 5 / 9
    df_total['MIN_C'] = (df_total['MIN'] - 32) * 5 / 9

    df_mensual = df_total.groupby('AÑO_MES').agg({
        'TEMP_C': 'mean',
        'MAX_C': 'mean',
        'MIN_C': 'mean'
    }).reset_index()

    # Guardar a un archivo CSV local
    df_mensual.to_csv("temperatura_mensual_eldorado_1980_2025.csv", index=False)
    print(f"Archivo guardado como: 'temperatura_mensual_eldorado.csv'")
    print(df_mensual.head())
else:
    print("\nNo se pudo descargar ningún dato.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_mensual['AÑO_MES_DT'] = df_mensual['AÑO_MES'].dt.to_timestamp()

plt.figure(figsize=(15, 7))
sns.lineplot(x='AÑO_MES_DT', y='TEMP_C', data=df_mensual)
plt.xlabel('Date')
plt.ylabel('Average Temperature (°C)')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.dates as mdates

# convert the datetime column to a numerical format
df_mensual['AÑO_MES_NUM'] = df_mensual['AÑO_MES_DT'].apply(mdates.date2num)

sns.lmplot(x='AÑO_MES_NUM', y='TEMP_C', data=df_mensual, ci=99, aspect=1.33, scatter_kws={'s': 10})

# make the x-axis readable
ax = plt.gca()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.YearLocator(5))
plt.gcf().autofmt_xdate()

plt.xlabel('Date')
plt.ylabel('Average Temperature (°C)')
plt.grid(True)
plt.show()

Solución exacta del problema de minimización - regresión lineal ideal:

$$\begin{equation}
Ax \approx y
\end{equation}$$

\begin{equation}
\hat{x} = \arg\min_{x} \|Ax - y\|_2^2
\end{equation}

$$\hat{x} = \left(A^{T}A\right)^{-1}A^{T}y
$$



**Demostración:**

$$\hat{x} = \arg\min_x \|Ax - y\|_2^2 \\
J(x) = \|Ax - y\|_2^2 \\
J(x) = (Ax - y)^T(Ax - y) \\
J(x) = x^T A^T A y - 2 y^T A x + y^T y \\
\nabla_x J(x) = 2A^T A x - 2A^T y \\
0 = 2A^T A x - 2A^T y \\
A^T A x = A^T y \\
x = (A^T A)^{-1} A^T y$$

In [ ]:
import numpy as np
from scipy import stats

# Prepare data for regression
x = df_mensual['AÑO_MES_NUM'].values
y = df_mensual['TEMP_C'].values

# Add a column of ones for the intercept term (matrix A)
A = np.vstack([x, np.ones(len(x))]).T


# Method 1: numpy.linalg.lstsq
# Returns: (coefficients, residuals, rank, singular_values)
coefficients_lstsq, residuals, rank, singular_values = np.linalg.lstsq(A, y, rcond=None)
slope_lstsq, intercept_lstsq = coefficients_lstsq
print(f"numpy.linalg.lstsq:  Slope = {slope_lstsq:.6f}, Intercept = {intercept_lstsq:.6f}")

# Method 2: Manual calculation using normal equation (beta = (A.T . A)^-1 . A.T . y)
# A is the design matrix (x with intercept), y is the target variable
b_manual = np.dot(np.dot(np.linalg.inv(np.dot(A.T, A)), A.T), y)
slope_manual, intercept_manual = b_manual
print(f"Manual (normal equation): Slope = {slope_manual:.6f}, Intercept = {intercept_manual:.6f}")

# Method 3: scipy.stats.linregress
# Directly returns (slope, intercept, r_value, p_value, std_err)
slope_linregress, intercept_linregress, r_value, p_value, std_err = stats.linregress(x, y)
print(f"scipy.stats.linregress: Slope = {slope_linregress:.6f}, Intercept = {intercept_linregress:.6f}")



In [ ]:
%%timeit
np.linalg.lstsq(A, y, rcond=None)

In [ ]:
%%timeit
np.dot(np.dot(np.linalg.inv(np.dot(A.T, A)), A.T), y)

In [ ]:
%%timeit
stats.linregress(x, y)

In [ ]:
# multiply by the number of days in a year (approx 365.25) and then by 10 for a decade.

# slope from numpy.linalg.lstsq
slope_degrees_per_day = slope_lstsq

days_in_a_year = 365.25
years_in_a_decade = 10

slope_degrees_per_decade = slope_degrees_per_day * days_in_a_year * years_in_a_decade

print(f"The temperature trend is {slope_degrees_per_decade:.4f} degrees Celsius per decade.")

In [ ]:
standard_error_per_day = std_err

standard_error_per_decade = standard_error_per_day * days_in_a_year * years_in_a_decade

print(f"The standard error of the temperature trend is {standard_error_per_decade:.4f} degrees Celsius per decade.")

Ahora calculemos manualmente el [intervalo de confianza](https://en.wikipedia.org/wiki/Confidence_interval).

In [ ]:

slope_linregress, intercept_linregress, r_value, p_value, std_err = stats.linregress(x, y)

# calculate predicted y values (regression line)
y_pred = slope_linregress * x + intercept_linregress

# calculate confidence intervals
# residual standard error (sigma)
n = len(x)
residuals = y - y_pred
rmse = np.sqrt(np.sum(residuals**2) / (n - 2)) # RSE

# calculate sum of squared differences for x
x_mean = np.mean(x)
Sxx = np.sum((x - x_mean)**2)

# standard error of the mean response for each x_i
se_y_pred = rmse * np.sqrt(1/n + (x - x_mean)**2 / Sxx)

# critical t-value for 95% confidence interval
confidence_level = 0.95
degrees_freedom = n - 2
t_critical = stats.t.ppf((1 + confidence_level) / 2, degrees_freedom)

# calculate upper and lower bounds of the confidence interval
lower_bound = y_pred - t_critical * se_y_pred
upper_bound = y_pred + t_critical * se_y_pred

plt.figure(figsize=(15, 7))

plt.scatter(df_mensual['AÑO_MES_DT'], y, label='Monthly Average Temperature', s=10, alpha=0.6)

plt.plot(df_mensual['AÑO_MES_DT'], y_pred, color='red', label=f'Linear Regression (Slope: {slope_linregress*365.25*10:.4f} °C/decade)')

# confidence interval band
plt.fill_between(df_mensual['AÑO_MES_DT'], lower_bound, upper_bound, color='red', alpha=0.2, label='95% Confidence Interval')

ax = plt.gca()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.YearLocator(5))
plt.gcf().autofmt_xdate()

plt.xlabel('Date')
plt.ylabel('Average Temperature (°C)')
plt.grid(True)
plt.legend()
plt.show()

Cambiemos el rango de tiempo para ver lo que ocurre con la calidad del ajuste.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

# filter df_mensual for the years 2005-2010
df_filtered = df_mensual[(df_mensual['AÑO_MES_DT'].dt.year >= 2005) & (df_mensual['AÑO_MES_DT'].dt.year <= 2008)].copy()

x_filtered = df_filtered['AÑO_MES_NUM'].values
y_filtered = df_filtered['TEMP_C'].values

slope_linregress_filtered, intercept_linregress_filtered, r_value_filtered, p_value_filtered, std_err_filtered = stats.linregress(x_filtered, y_filtered)

y_pred_filtered = slope_linregress_filtered * x_filtered + intercept_linregress_filtered

n_filtered = len(x_filtered)
residuals_filtered = y_filtered - y_pred_filtered
rmse_filtered = np.sqrt(np.sum(residuals_filtered**2) / (n_filtered - 2)) # Residual Standard Error

x_mean_filtered = np.mean(x_filtered)
Sxx_filtered = np.sum((x_filtered - x_mean_filtered)**2)

se_y_pred_filtered = rmse_filtered * np.sqrt(1/n_filtered + (x_filtered - x_mean_filtered)**2 / Sxx_filtered)

confidence_level = 0.95
degrees_freedom_filtered = n_filtered - 2
t_critical = stats.t.ppf((1 + confidence_level) / 2, degrees_freedom_filtered)

lower_bound_filtered = y_pred_filtered - t_critical * se_y_pred_filtered
upper_bound_filtered = y_pred_filtered + t_critical * se_y_pred_filtered

plt.figure(figsize=(15, 7))

plt.scatter(df_filtered['AÑO_MES_DT'], y_filtered, label='Monthly Average Temperature', s=10, alpha=0.6)

slope_degrees_per_decade_filtered = slope_linregress_filtered * 365.25 * 10
plt.plot(df_filtered['AÑO_MES_DT'], y_pred_filtered, color='red', label=f'Linear Regression (Slope: {slope_degrees_per_decade_filtered:.4f} °C/decade)')

plt.fill_between(df_filtered['AÑO_MES_DT'], lower_bound_filtered, upper_bound_filtered, color='red', alpha=0.2, label='95% Confidence Interval')

ax = plt.gca()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.YearLocator(1)) # Show every year
plt.gcf().autofmt_xdate()

plt.xlabel('Date')
plt.ylabel('Average Temperature (°C)')
plt.grid(True)
plt.legend()
plt.show()

## Estudio de curvas de luz de transientes - MANTRA

El proyecto [MANTRA](https://arxiv.org/abs/2006.13163) produjo curvas de luz de eventos transitorios astronómicos del Catalina Real-Time
Transient Survey (CRTS), cubriendo 33000 sq deg del cielo desde 2007, usando datos de Mt. Lemmon Survey (MLS), Catalina Sky Survey (CSS), Siding Spring Survey (SSS). Cada uno de los 4869 eventos transitorios en MANTRA ha sido asociado a algún tipo de evento astronómico por un ser humano. Vamos a tratar de hacer un clasificador supervisado de eventos transitorios.

In [ ]:
dfla=pd.read_csv("https://github.com/MachineLearningUniandes/MANTRA/raw/master/data/lightcurves/transient_labels.csv")
df=pd.read_csv("https://github.com/MachineLearningUniandes/MANTRA/raw/master/data/lightcurves/transient_lightcurves.csv")

Ordene de mayor a menor los tipos de eventos transitorios en esta base de datos.

Inspeccionemos una curva de luz

In [ ]:
from collections import Counter

ix=Counter(df.ID)
names=list(ix.keys())

i=320
name=names[i]
filt=df.ID==name
curve=(df[filt])
curve=curve.sort_values(by=['MJD'])
label=dfla.Classification[dfla.TransientID==float(name[6:])]
print(label,curve.shape)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(curve.MJD, curve.Mag, 'o', markersize=4)
plt.xlabel('Modified Julian Date (MJD)')
plt.ylabel('Magnitude')
plt.title(f'Light Curve for {label.iloc[0]} (ID: {name})')
plt.gca().invert_yaxis() # magnitudes
plt.show()

## Gaussian Processes (GP)

- Modelo probabilístico no paramétrico para regresión y clasificación.

- En lugar de aprender una función explícita, define una distribución sobre funciones.

- Cualquier conjunto finito de puntos tiene una distribución conjunta gaussiana.

- La inferencia se basa en una función de covarianza (*kernel*) que controla similitud entre puntos.

- El kernel incorpora supuestos sobre suavidad, periodicidad o escala de variación.

- Un Gaussian Process es una colección de variables aleatorias indexadas por \(x\), tal que cualquier subconjunto finito tiene distribución gaussiana conjunta.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process import kernels as k

# Prepare data for GPR
# Reshape MJD to a 2D array as required by scikit-learn
X = curve.MJD.values.reshape(-1, 1)
y = curve.Mag.values

# Define the kernel as specified by the user
kernel = k.RationalQuadratic(length_scale_bounds=(1e-7, 1e5)) + k.ConstantKernel() + k.RBF(length_scale=1000)

# Initialize Gaussian Process Regressor
gpr = GaussianProcessRegressor(kernel=kernel, alpha=0.1, random_state=0)

# Fit the GPR model to the data
gpr.fit(X, y)

# Predict over a denser range of MJD values for a smooth curve
X_pred = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)
y_mean_pred, y_std_pred = gpr.predict(X_pred, return_std=True)

# Plot the results
plt.figure(figsize=(12, 7))
plt.scatter(X, y, label='Original Data', s=15, alpha=0.7)
plt.plot(X_pred, y_mean_pred, color='red', label='GPR Mean Prediction')
plt.fill_between(X_pred.flatten(), y_mean_pred - 2 * y_std_pred, y_mean_pred + 2 * y_std_pred, color='red', alpha=0.2, label='95% Confidence Interval')

plt.xlabel('Modified Julian Date (MJD)')
plt.ylabel('Magnitude')
plt.title(f'Gaussian Process Regression for {label.iloc[0]} (ID: {name})')
plt.gca().invert_yaxis() # Invert y-axis for magnitudes
plt.grid(True)
plt.legend()
plt.show()